<a href="https://colab.research.google.com/github/jeevan841/Innomatics-Internship/blob/main/NLP_05_Finetune_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from datasets import load_dataset

dataset = load_dataset("a3lem/universal-dependencies-parquet", "en_ewt")
dataset

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2001 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/12544 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 2077
    })
    validation: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 2001
    })
    train: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 12544
    })
})

In [17]:
dataset["train"][0]

{'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001',
 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.',
 'ids': ['1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29'],
 'tokens': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'forces',
  'killed',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'Syrian',
  'border',
  '.'],
 'lemmas': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'force',
  'kill',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
 

In [18]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [19]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [20]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [21]:
all_labels = set()

for example in dataset["train"]:
    for tag in example["upos"]:
        all_labels.add(tag)

label_list = sorted(list(all_labels))

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

print(label_list)
print("Number of labels:", len(label_list))

['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_']
Number of labels: 18


In [22]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_to_id[label[word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [23]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

In [24]:
num_labels = len(label_list)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [25]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [27]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [29]:
import evaluate
seqeval = evaluate.load("seqeval")

In [30]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                current_preds.append(id_to_label[p_val])
                current_labels.append(id_to_label[l_val])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [31]:
training_args = TrainingArguments(
    output_dir="./pos_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
    load_best_model_at_end=True,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [33]:
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_val = tokenized_datasets["validation"].shuffle(seed=42).select(range(500))

In [34]:
print(small_train)
print(small_val)

Dataset({
    features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})
Dataset({
    features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [36]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PUNCT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171

{'eval_loss': 2.9187395572662354,
 'eval_model_preparation_time': 0.0035,
 'eval_precision': 0.060203283815480846,
 'eval_recall': 0.05166051660516605,
 'eval_f1': 0.05560570500090269,
 'eval_accuracy': 0.0668859649122807,
 'eval_runtime': 86.9744,
 'eval_samples_per_second': 5.749,
 'eval_steps_per_second': 0.724}

In [37]:
sentence = "John works at Google in California"
words = sentence.split()

inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2).squeeze().tolist()

word_ids = inputs.word_ids()
predicted_labels = []

previous_word_id = None
for pred, word_id in zip(predictions, word_ids):
    if word_id is not None and word_id != previous_word_id:
        predicted_labels.append(id_to_label[pred])
    previous_word_id = word_id

for word, label in zip(words, predicted_labels):
    print(f"{word}: {label}")

John: CCONJ
works: NOUN
at: PUNCT
Google: ADP
in: CCONJ
California: AUX
